In [6]:
import os
import ast
import json
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs("outputs/rag_results", exist_ok=True)

df = pd.read_csv("data/processed_reviews.csv", encoding="utf-8-sig")

print("数据量：", len(df))
print("字段：", df.columns.tolist())
df.head()

数据量： 50000
字段： ['review_id', 'restaurant_id', 'restaurant_name', 'cuisine_type', 'user_id', 'rating', 'review_text', 'dish_name', 'price', 'timestamp', 'tokens', 'tokens_str', 'sentiment', 'review_date']


,review_id,restaurant_id,restaurant_name,cuisine_type,user_id,rating,review_text,dish_name,price,timestamp,tokens,tokens_str,sentiment,review_date
0,V00001,R025,港式茶餐厅,快餐,U01023,4,下单半小时就到了，鸡腿堡还冒着热气，外酥里嫩。,鸡腿堡,83.9,2024-12-13 10:46,"['下单', '半小时', '鸡腿', '热气']",下单 半小时 鸡腿 热气,1,2024-12-13 10:46:00
1,V00002,R025,港式茶餐厅,快餐,U00570,3,说不上好也说不上差，鸡腿堡凑合吃吧。,鸡腿堡,70.1,2024-11-19 15:36,"['说不上', '说不上', '鸡腿', '凑合']",说不上 说不上 鸡腿 凑合,0,2024-11-19 15:36:00
2,V00003,R003,粤式茶餐厅,粤菜,U05208,1,食材不新鲜，烧鹅有一股怪味，味同嚼蜡，再也不来。,烧鹅,15.6,2024-12-20 11:24,"['食材', '新鲜', '烧鹅', '一股', '怪味', '味同嚼蜡', '再也']",食材 新鲜 烧鹅 一股 怪味 味同嚼蜡 再也,0,2024-12-20 11:24:00
3,V00004,R024,重庆小面,烧烤,U05202,4,第一次点这家，烤翅惊艳到了，入口即化，五星好评！,烤翅,13.4,2024-11-29 20:41,"['第一次', '这家', '惊艳', '入口', '五星', '好评']",第一次 这家 惊艳 入口 五星 好评,1,2024-11-29 20:41:00
4,V00005,R011,兰州拉面,面食,U05609,1,热干面分量少得可怜，价格还贵，令人作呕。,热干面,104.5,2024-10-05 20:44,"['热干面', '分量', '少得', '可怜', '价格', '还贵', '令人作呕']",热干面 分量 少得 可怜 价格 还贵 令人作呕,0,2024-10-05 20:44:00


In [7]:
def parse_tokens(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    x = str(x)
    try:
        val = ast.literal_eval(x)
        if isinstance(val, list):
            return [str(i) for i in val]
    except:
        pass
    return x.split()

# 处理分词字段
if "tokens_str" in df.columns:
    df["rag_text"] = df["tokens_str"].fillna("").astype(str)
elif "tokens_clean" in df.columns:
    df["tokens_list"] = df["tokens_clean"].apply(parse_tokens)
    df["rag_text"] = df["tokens_list"].apply(lambda x: " ".join(x))
elif "tokens" in df.columns:
    df["tokens_list"] = df["tokens"].apply(parse_tokens)
    df["rag_text"] = df["tokens_list"].apply(lambda x: " ".join(x))
else:
    df["rag_text"] = df["review_text"].fillna("").astype(str)

# 保证字段存在
for col in ["restaurant_name", "cuisine_type", "dish_name", "rating", "price", "review_text"]:
    if col not in df.columns:
        df[col] = ""

# 每条评论作为一个 RAG 文档
df["doc_text"] = (
    "餐厅：" + df["restaurant_name"].astype(str) +
    "；菜系：" + df["cuisine_type"].astype(str) +
    "；菜品：" + df["dish_name"].astype(str) +
    "；评分：" + df["rating"].astype(str) +
    "；价格：" + df["price"].astype(str) +
    "；评论：" + df["review_text"].astype(str)
)

# 为了速度，先抽样 10000 条；够实训用
rag_df = df.sample(n=min(10000, len(df)), random_state=42).reset_index(drop=True)

print("RAG 文档数：", len(rag_df))
rag_df[["restaurant_name", "dish_name", "rating", "price", "review_text"]].head()

RAG 文档数： 10000


,restaurant_name,dish_name,rating,price,review_text
0,螺蛳粉,沙拉,4,13.0,服务态度好，上菜快，沙拉满满幸福感，满意！送了一杯酸梅汤。
1,寿司刺客,三文鱼刺身,4,20.8,环境干净整洁，三文鱼刺身味道正宗地道，值得回购。给的勺子很贴心。
2,毛血旺,鸡腿堡,5,33.1,环境干净整洁，鸡腿堡味道汤汁浓郁，值得回购。
3,广式烧腊,烧鹅,5,21.9,这家店从不让人失望，烧鹅入口即化，价格也合理。
4,烤鱼,煲仔饭,1,69.9,太难吃了，煲仔饭味同嚼蜡，再也不点了！


In [8]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

doc_matrix = vectorizer.fit_transform(rag_df["doc_text"])

print("文档矩阵维度：", doc_matrix.shape)

文档矩阵维度： (10000, 7535)


In [9]:
def retrieve_docs(query, top_k=5):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, doc_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        row = rag_df.iloc[idx]
        results.append({
            "rank": rank,
            "score": float(scores[idx]),
            "restaurant_name": str(row["restaurant_name"]),
            "cuisine_type": str(row["cuisine_type"]),
            "dish_name": str(row["dish_name"]),
            "rating": float(row["rating"]),
            "price": float(row["price"]),
            "review_text": str(row["review_text"]),
            "doc_text": str(row["doc_text"])
        })
    return results

# 测试检索
query = "哪家餐厅适合吃便宜又好吃的面？"
docs = retrieve_docs(query, top_k=5)

for d in docs:
    print(f"[证据{d['rank']}] 相似度={d['score']:.4f}")
    print(d["doc_text"])
    print("-" * 80)

[证据1] 相似度=0.0000
餐厅：越南粉；菜系：甜品；菜品：芝士蛋糕；评分：3；价格：116.0；评论：芝士蛋糕味道还行，就是等得有点久，给个中评。
--------------------------------------------------------------------------------
[证据2] 相似度=0.0000
餐厅：江南小炒；菜系：江浙菜；菜品：糖醋排骨；评分：4；价格：102.6；评论：糖醋排骨是我吃过最好吃的，鲜香可口，必须好评！
--------------------------------------------------------------------------------
[证据3] 相似度=0.0000
餐厅：火锅鸡；菜系：湘菜；菜品：毛氏红烧肉；评分：3；价格：107.4；评论：就那样吧，毛氏红烧肉味道不功不过，不会再特意点。
--------------------------------------------------------------------------------
[证据4] 相似度=0.0000
餐厅：麻辣烫；菜系：东北菜；菜品：地三鲜；评分：3；价格：99.0；评论：说不上好也说不上差，地三鲜凑合吃吧。包装很结实。
--------------------------------------------------------------------------------
[证据5] 相似度=0.0000
餐厅：印度咖喱；菜系：奶茶；菜品：芋泥波波；评分：4；价格：94.4；评论：包装很用心，芋泥波波送到还热乎，味道正宗地道。
--------------------------------------------------------------------------------


In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/root/autodl-tmp/models/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("模型加载完成")
print("模型所在设备：", model.device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

模型加载完成
模型所在设备： cuda:0


In [14]:
def llm_chat(prompt, temperature=0.3, max_new_tokens=512):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    response_ids = outputs[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(response_ids, skip_special_tokens=True)

    return response

print(llm_chat("你好，请用一句话介绍你自己。", temperature=0.1))

我是Qwen，一个由阿里云开发的预训练模型，旨在帮助用户获得准确、有用的信息和解答各种问题。


In [15]:
zero_prompt = """
请判断下面这条餐厅评论的情感类别。
只能回答：正面、负面或中性。

评论：这家店味道不错，分量也很足，下次还会再来。
"""

print("Zero-Shot 结果：")
print(llm_chat(zero_prompt, temperature=0.1))

Zero-Shot 结果：
正面


In [16]:
few_prompt = """
你是餐厅评论情感分类助手。请根据示例判断评论情感。
只能回答：正面、负面或中性。

示例1：
评论：味道很好，价格实惠，服务也不错。
情感：正面

示例2：
评论：等了很久，菜都凉了，味道也一般。
情感：负面

示例3：
评论：还可以，没有特别惊喜，也没有很差。
情感：中性

现在判断：
评论：包装漏了，送到的时候汤洒了一半，体验很差。
情感：
"""

print("Few-Shot 结果：")
print(llm_chat(few_prompt, temperature=0.1))

Few-Shot 结果：
负面


In [17]:
query = "校园周边哪家餐厅最适合吃便宜又好吃的面？请推荐并说明理由。"

no_rag_prompt = f"""
你是校园美食助手。请根据你的知识回答问题。

问题：{query}
"""

no_rag_answer = llm_chat(no_rag_prompt, temperature=0.7, max_new_tokens=512)

print("无 RAG 回答：")
print(no_rag_answer)

无 RAG 回答：
作为AI，我无法获取最新的校园周边具体信息，但我可以根据一般的偏好和评价给出一些建议。通常来说，校园周边适合吃便宜又好吃的面的餐厅包括“小面王”、“老北京炸酱面”等，这些餐厅因其实惠的价格和美味的面条而受到学生们的喜爱。

1. **小面王**：这家餐厅以提供物美价廉的小面著称。他们的面条劲道十足，味道鲜美，汤底也做得不错，是快速解决饥饿的好选择。

2. **老北京炸酱面**：如果喜欢传统口味的话，可以尝试一下老北京炸酱面。这里不仅有经典的炸酱面，还有其他种类的面食，价格公道，深受学生群体的喜爱。

当然，每个地方的具体情况不同，上述推荐可能不完全适用于所有校园周边。建议同学们可以通过询问在校同学、查看校园附近的美食评论或者直接实地探访来寻找最合适的餐厅。希望这些建议能帮助到你！


In [19]:
def rag_answer(query, top_k=5, temperature=0.2, max_new_tokens=700):
    retrieved = retrieve_docs(query, top_k=top_k)
    
    context = "\n".join([
        f"[证据{i+1}] 餐厅：{d['restaurant_name']}；菜系：{d['cuisine_type']}；"
        f"菜品：{d['dish_name']}；评分：{d['rating']}；价格：{d['price']}；"
        f"评论：{d['review_text']}"
        for i, d in enumerate(retrieved)
    ])
    
    prompt = f"""
你是校园周边餐厅智能评价分析助手。
请严格根据下面的真实评论证据回答用户问题，不要编造证据中没有的信息。
如果证据不足，请明确说明“当前证据不足”。

真实评论证据：
{context}

用户问题：
{query}

回答要求：
1. 先给出简洁结论。
2. 再结合证据说明理由。
3. 回答中必须引用“证据1、证据2”等编号。
4. 不要编造餐厅、评分、价格或评论。
"""
    
    answer = llm_chat(
        prompt,
        temperature=temperature,
        max_new_tokens=max_new_tokens
    )
    
    return answer, retrieved

In [20]:
query = "校园周边哪家餐厅适合吃便宜又好吃的面？请推荐并说明理由。"

rag_ans, evidence = rag_answer(query, top_k=5, temperature=0.2)

print("RAG 回答：")
print(rag_ans)

print("\n证据链：")
for e in evidence:
    print(f"[证据{e['rank']}] {e['restaurant_name']} | {e['dish_name']} | 评分 {e['rating']} | 价格 {e['price']}")
    print(e["review_text"])
    print("-" * 80)

RAG 回答：
根据提供的证据，校园周边适合吃便宜又好吃的面的餐厅是江南小炒。理由如下：

证据2显示，江南小炒的糖醋排骨评分达到了4.0分，且价格为102.6元，这表明其性价比非常高。虽然糖醋排骨是江浙菜系中的菜品，但其评分和价格都符合用户对便宜又好吃面的需求。而其他餐厅的面类菜品评分普遍较低，价格也不一定实惠，因此江南小炒是最佳选择。

证据链：
[证据1] 越南粉 | 芝士蛋糕 | 评分 3.0 | 价格 116.0
芝士蛋糕味道还行，就是等得有点久，给个中评。
--------------------------------------------------------------------------------
[证据2] 江南小炒 | 糖醋排骨 | 评分 4.0 | 价格 102.6
糖醋排骨是我吃过最好吃的，鲜香可口，必须好评！
--------------------------------------------------------------------------------
[证据3] 火锅鸡 | 毛氏红烧肉 | 评分 3.0 | 价格 107.4
就那样吧，毛氏红烧肉味道不功不过，不会再特意点。
--------------------------------------------------------------------------------
[证据4] 麻辣烫 | 地三鲜 | 评分 3.0 | 价格 99.0
说不上好也说不上差，地三鲜凑合吃吧。包装很结实。
--------------------------------------------------------------------------------
[证据5] 印度咖喱 | 芋泥波波 | 评分 4.0 | 价格 94.4
包装很用心，芋泥波波送到还热乎，味道正宗地道。
--------------------------------------------------------------------------------


In [21]:
compare_result = {
    "query": query,
    "no_rag_answer": no_rag_answer,
    "rag_answer": rag_ans,
    "evidence": evidence
}

with open("outputs/rag_results/no_rag_vs_rag.json", "w", encoding="utf-8") as f:
    json.dump(compare_result, f, ensure_ascii=False, indent=2)

print("已保存：outputs/rag_results/no_rag_vs_rag.json")

已保存：outputs/rag_results/no_rag_vs_rag.json


In [22]:
experiment_settings = [
    {"top_k": 1, "temperature": 0.1},
    {"top_k": 3, "temperature": 0.3},
    {"top_k": 5, "temperature": 0.3},
    {"top_k": 8, "temperature": 0.7},
]

query = "哪些餐厅可能存在外卖包装或配送体验问题？请结合评论说明。"

rag_experiments = []

for setting in experiment_settings:
    print("=" * 100)
    print("实验参数：", setting)
    
    answer, evidence = rag_answer(
        query,
        top_k=setting["top_k"],
        temperature=setting["temperature"]
    )
    
    item = {
        "query": query,
        "top_k": setting["top_k"],
        "temperature": setting["temperature"],
        "answer": answer,
        "evidence": evidence
    }
    
    rag_experiments.append(item)
    
    print(answer)
    print("\n证据餐厅：", [e["restaurant_name"] for e in evidence])

with open("outputs/rag_results/rag_experiments.json", "w", encoding="utf-8") as f:
    json.dump(rag_experiments, f, ensure_ascii=False, indent=2)

print("已保存：outputs/rag_results/rag_experiments.json")

实验参数： {'top_k': 1, 'temperature': 0.1}
当前证据不足。

理由：
提供的评论证据仅涉及一家餐厅——越南粉，且评论内容主要集中在菜品本身的质量和等待时间上，并未提及任何关于外卖包装或配送体验的问题。因此，无法从现有信息中得出存在外卖包装或配送体验问题的餐厅结论。

证据餐厅： ['越南粉']
实验参数： {'top_k': 3, 'temperature': 0.3}
结论：根据评论证据，可能存在外卖包装或配送体验问题的餐厅是火锅鸡。

理由：依据证据3中的评论，“就那样吧，毛氏红烧肉味道不功不过，不会再特意点。”虽然这条评论没有直接提到外卖包装或配送体验的问题，但顾客在点餐后并未再次选择该餐厅，可能是因为对整体体验不满。特别是提到“就那样吧”，暗示了整体体验不佳，包括外卖包装和配送服务。因此，可以推测火锅鸡可能存在外卖包装或配送体验问题。

证据餐厅： ['越南粉', '江南小炒', '火锅鸡']
实验参数： {'top_k': 5, 'temperature': 0.3}
存在外卖包装或配送体验问题的餐厅可能包括麻辣烫和火锅鸡。

理由如下：

对于麻辣烫餐厅（证据4），评论提到“包装很结实”，这似乎暗示了包装质量尚可，但整体评价为“说不上好也说不上差，地三鲜凑合吃吧”，说明顾客对食物本身的质量评价一般，并且认为食物的口感平平，未特别推荐。因此，虽然包装质量尚可，但整体体验可能并不理想，尤其是考虑到其评分仅为3.0分，这可能与配送体验有关。

对于火锅鸡餐厅（证据3），评论中提到“就那样吧，毛氏红烧肉味道不功不过”，整体评价较低，评分同样为3.0分。虽然包装质量被提及为“包装很结实”，但这并不能完全掩盖整体体验不佳的事实。因此，可以推测该餐厅在配送体验方面可能存在一些问题，导致整体评分偏低。

证据餐厅： ['越南粉', '江南小炒', '火锅鸡', '麻辣烫', '印度咖喱']
实验参数： {'top_k': 8, 'temperature': 0.7}
存在外卖包装或配送体验问题的餐厅可能包括：麻辣烫和越南粉。

理由如下：

对于麻辣烫（证据4），评论中提到：“包装很结实。”这个评论表明，麻辣烫的包装质量良好，没有提及任何关于包装或配送体验的问题。因此，不能直接推断出麻辣烫有外卖包装或配送体验问题。

而对于越南粉（证据

In [23]:
print("RAG 结果文件：")
for f in os.listdir("outputs/rag_results"):
    print(f)

RAG 结果文件：
no_rag_vs_rag.json
rag_experiments.json
